# EEG Visual Simulation Project Report

## Reproduction of visual EEG effects using a transparent preprocessing and ICA workflow

This notebook documents the full analysis pipeline, the reasoning behind the
main preprocessing choices, and the interpretation of the resulting figures.
The goal is not only to show the final outputs, but also to make clear **why**
specific steps, parameters, and review decisions were used.

The project focuses on a qualitative reproduction of the main visual EEG
patterns reported in the target paper, especially:

- posterior alpha suppression,
- orientation-related structure in posterior responses,
- gamma-band effects,
- and retinotopy/model-based trends.

A central principle of this project was to keep the workflow interpretable.
For that reason, ICA review was done manually and the report emphasizes the
trade-offs and judgment behind the cleaning decisions.

---

## Table of contents

1. [Environment and helper code](#Environment-and-helper-code)  
2. [Dataset and subject selection](#Dataset-and-subject-selection)  
3. [Preprocessing pipeline](#Preprocessing-pipeline)  
4. [ICA review strategy](#ICA-review-strategy)  
5. [Subject-level observations](#Subject-level-observations)  
6. [Group-level analysis](#Group-level-analysis)  
7. [Main group figures](#Main-group-figures)  
8. [Interpretation of results](#Interpretation-of-results)  
9. [Methodological choices](#Why-I-made-these-choices)  
10. [Limitations and next steps](#Limitations-and-next-steps)  
11. [Final conclusion](#Final-conclusion)

## Environment and helper code

The next cell sets the project paths, loads JSON summary files, and defines a
small plotting helper used throughout the notebook. This keeps the rest of the
report cleaner and makes the displayed figures reproducible from the saved
outputs directory.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt


ROOT = Path("/Volumes/personal/EEG/eeg_visual_simulation_lac")
OUTPUTS = ROOT / "Dataset" / "outputs"
SUBJECTS = ["sub-01", "sub-02", "sub-03"]


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def show_image(path: Path, figsize=(10, 6), title: str | None = None):
    if not path.exists():
        print(f"Missing image: {path}")
        return
    image = mpimg.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()




## Dataset and Subject Selection

I used the dataset in:

- `/Volumes/personal/EEG/ds006547`

The initial full pipeline was run for three subjects:

- `sub-01`
- `sub-02`
- `sub-03`

I started with a small subject set because this made it possible to debug the
pipeline carefully, inspect ICA decisions manually, and evaluate how close the
outputs were to the paper before scaling further.

## Important Data-Handling Detail

The BrainVision files are stored through `git-annex` symlinks. Early in the
project, resolving the `.vhdr` path caused the scripts to follow the annex
object path instead of the subject path, which broke `.vmrk` lookup. I fixed
the scripts so they preserve the subject-level BrainVision path instead of
resolving it.



## Preprocessing Pipeline

The preprocessing pipeline was implemented in `Dataset/Scripts/preprocess.py`
and followed this logic:

1. Load the BrainVision recording.
2. Apply montage information.
3. Detect bad channels.
4. Filter data for ICA preparation.
5. Fit ICA on event-centered data.
6. Save the ICA model and a review template JSON.

### Why ICA Was Done Manually

I did not want automatic component rejection because ICA decisions are highly
consequential and easy to over-apply. The repository was set up so that:

- ICA is fit automatically,
- component plots are generated automatically,
- but keep/reject decisions are made manually in a JSON file.

This was important because a major risk in EEG cleaning is accidentally
removing neural signal instead of artifact.

### Key Parameter Choices

- ICA components were capped by the number of usable EEG channels so the fit
  stays valid across subjects.
- Harmonic notch filtering was extended beyond a single 60 Hz notch to include
  harmonics (for example 60, 120, 180 Hz up to Nyquist).
- Gamma summary power was made more robust by avoiding the 60 Hz bin and using
  sub-bands `40-55 Hz` and `65-80 Hz`.

### Why These Changes Were Necessary

The single biggest problem during analysis was that residual line noise still
affected gamma-band results. Alpha results were more stable, but gamma and ERSP
outputs were visibly degraded when 60 Hz contamination remained in the spectra.
Strengthening harmonic notch filtering and avoiding the 60 Hz bin in gamma
summaries improved the robustness of the downstream results.



## ICA Review Strategy

ICA was reviewed manually subject by subject using:

- topographies,
- component property plots,
- spectral shape,
- and plausibility of neural versus artifact origin.

### Decision Rule

I kept components that looked like broad, smooth posterior or plausible
cortical sources, especially when they were consistent with visual cortex
activity.

I rejected components that looked like:

- frontal blink/eye artifacts,
- temporal muscle artifacts,
- edge-dominant focal hotspots,
- inferior neck/jaw contamination,
- or diffuse non-dipolar/global artifact patterns.

### Why This Was Important

A major lesson from the project was that ICA choices strongly change the final
physiological interpretability of the results. Early conservative choices that
kept too many mixed or temporal components often produced noisier gamma and ERSP
outputs. Later, stricter posterior-focused selections improved the quality of
the outputs.



## Subject-Level Observations

The pipeline was tested and iteratively refined on `sub-01`, `sub-02`, and
`sub-03`.

Broadly:

- `sub-01`: pipeline validation subject, but still noisy.
- `sub-02`: usable but less convincing than hoped for orientation/gamma.
- `sub-03`: strongest of the first three subjects, but still noisier than the
  paper at the single-subject ERSP level.

This led to an important interpretation choice: I treated single-subject ERSP
outputs mainly as quality-control and exploratory evidence, rather than as the
main basis for claiming replication of the paper.



### Available subject summaries

This quick check verifies which of the processed subjects currently have saved
band-power summary files available in the outputs folder. It is a simple sanity
check before moving to the group-level interpretation.

In [ ]:
subject_summaries = {}
for subject in SUBJECTS:
    subject_dir = OUTPUTS / subject
    summary_path = subject_dir / f"{subject}_ses-01_task-visual_eeg-band_power_summary.json"
    if summary_path.exists():
        subject_summaries[subject] = load_json(summary_path)

print("Subjects with band-power summaries:", sorted(subject_summaries))




## Group-Level Analysis

After subject-level debugging, I moved to group-level analyses because the
paper's claims should be evaluated primarily through cross-subject consistency,
not by expecting a perfect ERSP map from a single subject.

The main group scripts used were:

- `Dataset/Scripts/grand_average_analysis.py`
- `Dataset/Scripts/group_topomaps.py`
- `Dataset/Scripts/retinotopy_model_fit.py`

These scripts summarized the processed subject outputs into:

- grand-average alpha and gamma topographies,
- condition-wise topographic summaries,
- orientation tuning summaries,
- and model-fit comparisons between linear and divisive normalization accounts.



### Load saved group-level summaries

These JSON files contain the numerical summaries produced by the group scripts.
Loading them here makes it possible to connect the figures to the stored outputs
instead of treating the plots as standalone images without context.

In [ ]:
grand_average_summary_path = OUTPUTS / "grand_average_summary.json"
group_topomap_summary_path = OUTPUTS / "group_topomap_summary.json"
model_fit_summary_path = OUTPUTS / "retinotopy_model_fit_summary.json"

grand_average_summary = load_json(grand_average_summary_path) if grand_average_summary_path.exists() else {}
group_topomap_summary = load_json(group_topomap_summary_path) if group_topomap_summary_path.exists() else {}
model_fit_summary = load_json(model_fit_summary_path) if model_fit_summary_path.exists() else {}

print("Grand-average summary keys:", sorted(grand_average_summary.keys()))
print("Group topomap summary keys:", sorted(group_topomap_summary.keys()))
print("Model-fit summary keys:", sorted(model_fit_summary.keys()))




## Main group figures

The figures below are the main visual outputs I rely on in the interpretation.
They were chosen because they best summarize whether the replication succeeded at
the level of the paper's main qualitative claims.

For grading, the key point is not just that the figures exist, but what each one
is intended to test:

- **Grand-average topomaps** check whether alpha and gamma effects appear in the
  expected posterior regions.
- **Condition-wise alpha maps** test whether the effect varies systematically
  across visual conditions.
- **Orientation tuning curves** summarize whether the posterior responses carry
  interpretable orientation structure.
- **Retinotopy/model-fit plots** test whether the extracted summaries support the
  expected modeling trends.

In [ ]:
show_image(
    OUTPUTS / "grand_average_topomaps.png",
    figsize=(12, 5),
    title="Grand Average Alpha and Gamma Topomaps",
)

show_image(
    OUTPUTS / "Condition_Wise_Alpha_Topomaps_rebuilt.png",
    figsize=(12, 14),
    title="Condition-Wise Alpha Topomaps",
)

show_image(
    OUTPUTS / "Orientation_Tuning_Curves.png",
    figsize=(10, 4),
    title="Posterior Orientation Tuning Curves",
)

show_image(
    OUTPUTS / "retinotopy_model_fit_errors.png",
    figsize=(14, 5),
    title="Retinotopy Model Fit Errors",
)




## Interpretation of results

## Interpretation of Group Results

### 1. Alpha Replication

Alpha was the strongest and cleanest part of the project.

The grand-average alpha topography shows broad posterior suppression, which is
qualitatively consistent with the expected visual alpha response. The condition-
wise alpha maps also show systematic variation rather than flat or random
structure.

This is the part of the replication I consider the most convincing.

### 2. Gamma Replication

Gamma was weaker and noisier than alpha throughout the project.

Even after improving harmonic notch filtering and avoiding the 60 Hz bin in the
gamma summary, gamma remained sensitive to residual noise and outlier channels.
Group-level gamma patterns are suggestive but not as clean or as stable as the
paper.

I therefore interpret the gamma findings as partially supportive but weaker than
the target paper.

### 3. Orientation Tuning

The posterior tuning curves show that gamma varies with orientation more than
alpha, while alpha remains broader and more negative overall. This is
qualitatively in the direction expected from the paper.

However, the ERSP-based orientation ANOVA maps remained fragmented and speckled.
The orientation effect was therefore more convincing in the summarized tuning
curves than in the cluster-based ERSP maps.

### 4. Retinotopy Model Comparison

The model-fit comparison is promising. In the current outputs, the divisive
normalization model generally performs better than the linear model across the
retinotopy groupings shown in the figure.

This does not prove a perfect replication, but it does suggest that the overall
direction of the model comparison is meaningful and aligned with the paper's
conceptual interpretation.



## What Matched the Paper Well

The main qualitative findings that were reproduced reasonably well are:

- broad posterior alpha suppression,
- orientation dependence in posterior summaries,
- and a promising group-level divisive-normalization advantage over a linear
  model.

These are the parts of the project I would defend most confidently.



## What Matched the Paper Less Well

The weakest part of the reproduction was the gamma/ERSP side:

- group gamma topographies were still noisier than desired,
- subject-level gamma remained sensitive to residual line noise and outliers,
- and the ERSP ANOVA maps produced many small scattered clusters instead of a
  clean coherent effect region.

Because of this, I would not claim a strong one-to-one replication of the
paper's gamma/ERSP figures. Instead, I would say that the project provides a
partial qualitative replication with alpha being strongest, and gamma being
suggestive but not cleanly reproduced.



## Why I Made These Choices

I made the following major methodological choices intentionally:

- **Manual ICA review instead of automatic rejection**
  because I wanted to avoid accidental removal of neural components.

- **Posterior-focused keep strategy for ICA**
  because the paper is primarily about visual alpha/gamma effects, not about
  retaining every plausible cortical source.

- **Harmonic notch filtering**
  because a single 60 Hz notch was not sufficient for stable gamma analysis.

- **Gamma sub-bands excluding 60 Hz**
  because including the line-noise bin made the gamma summary less trustworthy.

- **Moving to group-level interpretation**
  because single-subject ERSP results were too noisy to serve as the main
  replication criterion.



## Limitations

The main limitations of the project are:

1. Only a small number of subjects were processed at the current stage.
2. Gamma remains sensitive to residual noise.
3. The ERSP cluster method is relatively simple and can produce fragmented maps.
4. Manual ICA review introduces reasonable but subjective judgment.
5. Some subject-level outputs remain noisier than ideal despite cleaning.



## What I Would Improve Next

If I continued this project, I would prioritize:

1. processing more subjects,
2. strengthening group-level statistical analysis,
3. improving ERSP cluster inference,
4. adding explicit group exclusions for unusually noisy subjects/channels,
5. and refining the gamma summary further if residual line-noise effects remain.

In other words, the next biggest improvement would likely come from scaling and
stronger group inference, not from endlessly tuning one subject at a time.



## Final Conclusion

This project achieved a partial qualitative replication of the target paper.

The strongest replicated result is broad posterior alpha suppression. The
orientation and retinotopy analyses are partially supportive and show structured
effects. The model-fit comparison is promising and points in the expected
direction for divisive normalization. The weakest part is the gamma/ERSP
replication, which remains noisier and less coherent than the paper.

Overall, I understand the full analysis pipeline and can justify the decisions
made at each stage, even where the results remained imperfect. The report is
meant to make that reasoning transparent.



## Reproducibility note

This notebook is designed as a report notebook rather than a raw lab notebook.
It assumes that the pipeline has already been executed and that the figures and
summary JSON files have been written into the `Dataset/outputs` directory. The
advantage of this format is that it keeps the narrative focused on decisions,
reasoning, and interpretation while still showing exactly which saved outputs
the conclusions are based on.

In [ ]:
print("Report file ready:", ROOT / "submission_report.py")

